In [ ]:
import polars as pl
_=pl.Config.set_tbl_cols(100000)
_=pl.Config.set_tbl_rows(10000)
_=pl.Config.set_tbl_width_chars(10000)
_=pl.Config.set_fmt_str_lengths(10000)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
import seaborn as sns
import pickle
from sklearn.metrics import matthews_corrcoef
from pathlib import Path

In [ ]:
from pathlib import Path

def get_path(cell_line):
    """
    Returns the full path to the cell line data directory,
    relative to this script's location.
    
    Args:
        cell_line (str): The name of the cell line (e.g. "K562" or "HepG2").
    
    Returns:
        str: Full path to the data file for the given cell line.
    """
    data_dir = Path("../../06_first_order_SHAP_analysis/outputs/FINAL_AVERAGE_SHAP_CACHE")

    return str(data_dir / f"{cell_line}_all-data.feather")

In [ ]:
# Read in data for both cell lines

data_path_HepG2 = get_path("HepG2")
all_model_data_HepG2 = pl.read_ipc(data_path_HepG2)

In [ ]:
data_path_K562 = get_path("K562")
all_model_data_K562 = pl.read_ipc(data_path_K562)

## Data generation

In [ ]:
def rbp_activity_heatmap_data(cell_line, data):
    
    print(cell_line)
    
    # Filter massive dataframe for KD samples, FDR <= 0.1, abs[DeltaPSI] > 0

    filtered_data = (
        data
        .filter(
            (data["Sample Name"].str.contains("_KD-", literal=True))
            & (data["FDR"] <= 0.1)
            & (data["DeltaPSI"].abs() > 0)
        )    
    )

    # Columns 
    columns_to_keep = [
        "RBP_KD_Target",
        "Sample Name",
        "FDR",
        "DeltaPSI",
        "has_RBP_KD_1",
        "has_RBP_KD_2",
        "has_RBP_KD_3",
        "has_RBP_KD_4",
        "has_RBP_KD_5",
        "has_RBP_KD_6",
        "Raw P-Val",
        "rMATS Event ID"
    ]
    
    rbp_targets = (
        filtered_data["RBP_KD_Target"]
        .unique()
        .sort()
        .to_list()
    )
    
    positions = [1,2,3,4,5,6]

    results = []  # collect all results here

    for rbp in rbp_targets:
        for pos in positions:

            rbp_specific_data = (
            filtered_data
            .filter(
                (pl.col("RBP_KD_Target") == rbp) & (pl.col(f"has_RBP_KD_{pos}") == True))
            .unique(subset=["rMATS Event ID"])) # gets in-silico KD rows per RBP per position

            # Drop unneccesary columns
            rbp_specific_data = rbp_specific_data.select(columns_to_keep)

            # Convert to pandas
            df = rbp_specific_data.to_pandas()

            # Prevents error for no data (SAFB or IGF..1)
            data = df["DeltaPSI"]

            if data.empty:
                continue 

            # DPSI conversion
            df["DeltaPSI"] = df["DeltaPSI"] * -1

            # Calculate the number of positive dPSI and the number of negative dPSI

            num_pos = (df["DeltaPSI"] > 0).sum()
            num_neg = (df["DeltaPSI"] < 0).sum()

            # Do not include rows plots with less than 10 sig. points 

            if (num_pos + num_neg) < 10:
                continue 

            # Normalized Difference Score
            norm_diff = (num_pos - num_neg) / (num_pos + num_neg)

            # Append results
            results.append({
                "RBP": rbp,
                "position": pos,
                "num_pos": num_pos,
                "num_neg": num_neg,
                "norm_diff": norm_diff
            })

    
    results_df = pd.DataFrame(results)

    # Pivot the dataframe to have positions as columns, RBPs as rows, and score as values
    heatmap_data = results_df.pivot(index='RBP', columns='position', values='norm_diff')

    return heatmap_data, results_df

In [ ]:
# Get heatmap data and format for plot

HepG2_heatmap, HepG2_results_df = rbp_activity_heatmap_data('HepG2', all_model_data_HepG2)
K562_heatmap, K562_results_df = rbp_activity_heatmap_data('K562', all_model_data_K562)

HepG2_heatmap.columns.name = None
K562_heatmap.columns.name = None

# Add Cell Line column
K562_heatmap["Cell Line"] = "K562"
HepG2_heatmap["Cell Line"] = "HepG2"

final_heatmap = pd.concat([K562_heatmap, HepG2_heatmap])

In [ ]:
final_heatmap.to_csv("rbp_activity_heatmap.csv")

## Heatmap plot

In [ ]:
def plot_rbp_activity_heatmap_test(heatmap_data):

    plt.style.use("../../paper.mplstyle")

    # Split data
    k562_data = heatmap_data[heatmap_data["Cell Line"] == "K562"].drop(columns="Cell Line")
    hepg2_data = heatmap_data[heatmap_data["Cell Line"] == "HepG2"].drop(columns="Cell Line")

    # Get union of RBPs across both cell lines
    all_rbps = sorted(set(k562_data.index).union(set(hepg2_data.index)))
    
    # Reindex both datasets to the full RBP list
    k562_data = k562_data.reindex(all_rbps)
    hepg2_data = hepg2_data.reindex(all_rbps)

    # Ensure identical RBP order
    hepg2_data = hepg2_data.reindex(k562_data.index)

    cell_line_data = [("HepG2", hepg2_data), ("K562", k562_data)]

    top_n = len(k562_data)

    vmin = min(df.min().min() for _, df in cell_line_data)
    vmax = max(df.max().max() for _, df in cell_line_data)

    fig, axes = plt.subplots(
        1, 2,
        figsize=(6.5, 0.15 * top_n),  # smaller row height
        dpi=300,
        sharex=True,
        sharey=True
    )

    cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.6])

    for idx, (cell_line, heatmap) in enumerate(cell_line_data):

        sns.heatmap(
            heatmap,
            ax=axes[idx],
            cmap="bwr",
            center=0,
            vmin=vmin,
            vmax=vmax,
            cbar=(idx == 0),
            cbar_ax=(cbar_ax if idx == 0 else None),
            linewidths=0.4,
            linecolor="black"
        )

        axes[idx].set_title(cell_line, fontsize=12, pad=5)
        axes[idx].set_xlabel("")
        axes[idx].set_ylabel("")

        null_rows, null_cols = np.where(heatmap.isnull())
        axes[idx].scatter(
            null_cols + 0.5,
            null_rows +0.5,
            s=12,
            color="lightgray",
            marker="o",
            linewidths=0,
            zorder=10,
        )

    # Set RBP labels

    # PROCEED WITH CAUTION IF RE-MAKING THIS FIGURE WITH DIFFERENT DATA
    
    axes[0].set_yticks(np.arange(len(all_rbps)) + 0.5)
    axes[0].set_yticklabels(all_rbps, fontsize=7, rotation=0)

    axes[1].tick_params(axis='y', left=True, labelleft=False)
    axes[0].tick_params(axis='x', labelsize=12)
    axes[1].tick_params(axis='x', labelsize=12)

    label = r'Activity'

    cbar_ax.set_title(label, fontsize=10, pad=10)
    cbar_ax.tick_params(labelsize=10)

    fig.supxlabel("Position", fontsize=14, x=0.55, y=0.02,  fontweight="bold")

    plt.subplots_adjust(left=0.22, right=0.88, top=0.95, bottom=0.08, wspace=0.05)

    plt.savefig("rbp_activity_heatmap.pdf", bbox_inches='tight')
    
    plt.show()

In [ ]:
plot_rbp_activity_heatmap_test(final_heatmap)

## Scatterplot

In [ ]:
# Plot Scatterplot with MCC

def scatterplot(cell_line, data, results_df):
    
    print(cell_line)
    
    df = results_df.rename(columns={"position": "Position"})

    glossary_data_path = f"../../06_first_order_SHAP_analysis/outputs/specialized_global_SHAP/signed_local_SHAP_mean_bound_only_unique_binding.pkl"

    with open(glossary_data_path, 'rb') as f:
        glossary_raw = pickle.load(f)
    keys = list(glossary_raw.keys())
    
    glossary_map = {keys[0]: pd.DataFrame(glossary_raw[keys[0]]), keys[1]: pd.DataFrame(glossary_raw[keys[1]])}
    for cl in glossary_map:
        glossary_map[cl].columns.name = None

    glossary_data = glossary_map[cell_line]
    glossary_data = glossary_data.reset_index()
    
    # Melt glossary from wide to long so it matches activity data structure

    glossary_long = glossary_data.melt(id_vars="Position", var_name="RBP", value_name="shap")

    # Merge on position + RBP

    merged = df.merge(glossary_long, on=["RBP", "Position"], how="left")
    
    merged["pred_sign"] = np.sign(merged["shap"])
    merged["true_sign"] = np.sign(merged["norm_diff"])
    
    # Drop zeros
    merged_mcc = merged[(merged["pred_sign"] != 0) & (merged["true_sign"] != 0)]
    
    # Fisher's Exact Test (2x2 contingency table on same sign-filtered data)
    # Binarize: positive sign = 1, negative sign = 0
    pred_pos = merged_mcc["pred_sign"] == 1
    pred_neg = merged_mcc["pred_sign"] == -1
    true_pos = merged_mcc["true_sign"] == 1
    true_neg = merged_mcc["true_sign"] == -1
    
    tp = (pred_pos & true_pos).sum()   # Q1: both positive
    fp = (pred_pos & true_neg).sum()   # Q4: pred+, true-
    fn = (pred_neg & true_pos).sum()   # Q2: pred-, true+
    tn = (pred_neg & true_neg).sum()   # Q3: both negative
    
    contingency_table = [[tp, fp], [fn, tn]]
    
    # Scatterplot
    conditions = [
        (merged["shap"] > 0) & (merged["norm_diff"] > 0),   # Q1
        (merged["shap"] < 0) & (merged["norm_diff"] > 0),   # Q2
        (merged["shap"] < 0) & (merged["norm_diff"] < 0),   # Q3
        (merged["shap"] > 0) & (merged["norm_diff"] < 0),   # Q4
    ]
    colors = ["red", "gray", "blue", "gray"]
    merged["quadrant_color"] = np.select(conditions, colors, default="black")
    
    # Count quadrant points
    q1 = ((merged["shap"] > 0) & (merged["norm_diff"] > 0)).sum()
    q3 = ((merged["shap"] < 0) & (merged["norm_diff"] < 0)).sum()
    
    total_points = len(merged)
    
    conc = ((q1 + q3) / total_points) if q3 != 0 else np.nan
    conc = conc * 100
    
    label = r'Avg. SHAP'
    score = r'Activity'
    plt.style.use("../../paper.mplstyle")
    plt.figure(figsize=(8, 6))
    plt.scatter(merged["shap"], merged["norm_diff"], c=merged["quadrant_color"], alpha=0.5, rasterized=True)
    plt.ylim(-1.05, 1.05)
    
    # Add horizontal & vertical lines at 0 to create 4 quadrants
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.axvline(0, color="black", linestyle="--", linewidth=1)
    
    # Labels & title
    plt.xlabel(label, fontweight='bold', fontsize=21, labelpad=13)
    plt.ylabel(score, fontweight='bold', fontsize=21, labelpad=13)
    ax = plt.gca()
    ax.text(0.02, 0.97, f"{cell_line}", transform=ax.transAxes,
            fontsize=22, fontweight='bold', va='top', ha='left')
    
    ax.text(
        0.98, 0.03,
        f"Sign Concordance: {conc:.1f}%",
        transform=ax.transAxes,
        fontsize=15,
        va='bottom',
        ha='right',
        bbox=dict(
            boxstyle="round,pad=0.3",
            facecolor="none",
            edgecolor="lightgray",
            alpha=0.6
        )
    )
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)
    plt.savefig(f"{cell_line}_scatterplot.pdf", bbox_inches='tight', dpi=200)
    plt.show()

In [ ]:
scatterplot('K562', all_model_data_K562, K562_results_df)

In [ ]:
scatterplot('HepG2', all_model_data_HepG2, HepG2_results_df)